# Transformer PyTorch — Model Verification & Tokenizer Training

This notebook serves two purposes:

### 1. Module Verification
Testing each PyTorch module to ensure correct output shapes 
and behavior before training:
- `MultiHeadAttention`
- `FeedForward`
- `LayerNorm`
- `Encoder`
- `Decoder`
- `Transformer`

### 2. Tokenizer Training
Training SentencePiece BPE tokenizers on the opus100 Korean-English dataset:
- Korean tokenizer (vocab size: 16,000)
- English tokenizer (vocab size: 16,000)

Trained tokenizer models are saved to `data/processed/` 
and reused in `02_pretrain.ipynb` (coceptual code) and `03_finetune.ipynb`.

## 1. Module Verification

In [1]:
import torch
import os
import sys
sys.path.append('./src')

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

2.11.0+cu130
True
NVIDIA GeForce RTX 4080 SUPER
VRAM: 16.0 GB


In [4]:
from attention import MultiHeadAttention

d_model, num_heads, batch, seq_len = 512, 8, 2, 10

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch, seq_len, d_model)
output, attn_weights = mha(x, x, x)

print("Output shape:", output.shape)       # (2, 10, 512)
print("Attn weights:", attn_weights.shape) # (2, 8, 10, 10)

Output shape: torch.Size([2, 10, 512])
Attn weights: torch.Size([2, 8, 10, 10])


In [5]:
from feedforward import FeedForward

d_model, d_ff = 512, 2048
ff = FeedForward(d_model, d_ff)
x = torch.randn(2, 10, d_model)
output = ff(x)

print("Output shape:", output.shape)  # (2, 10, 512)

Output shape: torch.Size([2, 10, 512])


In [6]:
from layer_norm import LayerNorm

ln = LayerNorm(d_model)
x = torch.randn(2, 10, d_model)
output = ln(x)

print("Output shape:", output.shape)          # (2, 10, 512)
print("Mean (≈0):", output.mean(-1).mean().item())
print("Std  (≈1):", output.std(-1).mean().item())

Output shape: torch.Size([2, 10, 512])
Mean (≈0): 8.032657317080805e-10
Std  (≈1): 0.9999990463256836


In [7]:
from encoder import Encoder

encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 10, 512)
output = encoder(x)

print("Output shape:", output.shape)  # (2, 10, 512)

Output shape: torch.Size([2, 10, 512])


In [8]:
from decoder import Decoder

decoder = Decoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)

encoder_output = torch.randn(2, 10, 512)  # from encoder
tgt = torch.randn(2, 7, 512)             # target sequence

output = decoder(tgt, encoder_output)

print("Output shape:", output.shape)  # (2, 7, 512)

Output shape: torch.Size([2, 7, 512])


In [9]:
from transformer import Transformer

model = Transformer(src_vocab_size=10000, tgt_vocab_size=10000)
model = model.cuda()

# Dummy token indices
src = torch.randint(0, 10000, (2, 10)).cuda() 
tgt = torch.randint(0, 10000, (2, 7)).cuda() 

src_mask = model.make_src_mask(src, pad_idx = 0)
tgt_mask = model.make_tgt_mask(tgt, pad_idx = 0)

output = model(src, tgt, src_mask, tgt_mask)

print("Output shape:", output.shape)  # (2, 7, 10000)

Output shape: torch.Size([2, 7, 10000])


## 2. Tokenizer Training

In [2]:
from datasets import load_dataset

dataset = load_dataset("opus100", "en-ko")
print(dataset)
print(dataset['train'][0])

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
{'translation': {'en': "They're shaped like a bus.", 'ko': '할머니처럼 만들었지만.. ? 엉망이지만..'}}


In [4]:
os.makedirs('./data/raw', exist_ok=True)

# Save Korean and English sentences to text files for tokenizer training
with open('./data/raw/ko.txt', 'w', encoding='utf-8') as f:
    for item in dataset['train']:
        f.write(item['translation']['ko'] + '\n')

with open('./data/raw/en.txt', 'w', encoding='utf-8') as f:
    for item in dataset['train']:
        f.write(item['translation']['en'] + '\n')

print(f"Total sentences: {len(dataset['train'])}")

Total sentences: 1000000


In [5]:
with open('./data/raw/ko.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    last_5_lines = lines[-10:]
    
for i, line in enumerate(last_5_lines):
    print(f"Line {i+1}: {repr(line)}")

Line 1: '씨발!\n'
Line 2: '- 도와줘요!\n'
Line 3: '지금은 바하 집에서 지내고 계시지\n'
Line 4: '-아이를 위해서\n'
Line 5: '저 가족이 위험해보였으니까\n'
Line 6: '- 의학적인 게 아니라 사적인 거야\n'
Line 7: '케틀린은 좋은 엄마같더라\n'
Line 8: '[Clattering]\n'
Line 9: '그리고 그 결심을 렉시와 얼마나 함께 하고 싶었는지도\n'
Line 10: '두고 봐!\n'


In [6]:
with open('./data/raw/en.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
    last_5_lines = lines[-10:]
    
for i, line in enumerate(last_5_lines):
    print(f"Line {i+1}: {repr(line)}")

Line 1: 'Shit!\n'
Line 2: '- Help!\n'
Line 3: "She's staying at the Baja house now.\n"
Line 4: "- For the kid's sake,\n"
Line 5: 'I saved the family.\n'
Line 6: "- This isn't medical it is personal.\n"
Line 7: 'Kathleen seems like a good mom.\n'
Line 8: '[Clattering]\n'
Line 9: 'D you know how much I wanted to take that step with lexie.\n'
Line 10: "I'll show you!\n"


In [ ]:
# ko_sentences = [item['translation']['ko'] for item in dataset['train'] if item['translation']['ko']]
# en_sentences = [item['translation']['en'] for item in dataset['train'] if item['translation']['en']]

In [10]:
from tokenizer import Tokenizer

sys.path.append('./src')
os.makedirs('./data/processed', exist_ok=True)

# Train Korean tokenizer
ko_tokenizer = Tokenizer()
ko_tokenizer.train(
    input_file='./data/raw/ko.txt',
    # sentences = ko_sentences,
    model_prefix='./data/processed/ko_tokenizer',
    vocab_size=8000
)

# Train English tokenizer
en_tokenizer = Tokenizer()
en_tokenizer.train(
    input_file='./data/raw/en.txt',
    # sentences = en_sentences,
    model_prefix='./data/processed/en_tokenizer',
    vocab_size=8000
)

--- Starting SentencePiece training for ./data/processed/ko_tokenizer ---
Tokenizer trained. Vocab size: 8000
--- Starting SentencePiece training for ./data/processed/en_tokenizer ---
Tokenizer trained. Vocab size: 8000


In [12]:
from tokenizer import Tokenizer

ko_tokenizer = Tokenizer()
ko_tokenizer.load('./data/processed/ko_tokenizer.model')

en_tokenizer = Tokenizer()
en_tokenizer.load('./data/processed/en_tokenizer.model')

# Test Korean tokenizer
text_ko = "나는 학교에 갑니다"
encoded = ko_tokenizer.encode(text_ko)
decoded = ko_tokenizer.decode(encoded)
print(f"Original : {text_ko}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")

# Test English tokenizer
text_en = "I go to school"
encoded = en_tokenizer.encode(text_en)
decoded = en_tokenizer.decode(encoded)
print(f"Original : {text_en}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")

Tokenizer loaded. Vocab size: 8000
Tokenizer loaded. Vocab size: 8000
Original : 나는 학교에 갑니다
Encoded  : [2, 611, 1881, 6710, 1553, 273, 3]
Decoded  : 나는 학교에 갑니다
Original : I go to school
Encoded  : [2, 272, 396, 287, 1495, 3]
Decoded  : I go to school


In [3]:
from dataset import TranslationDataset, get_dataloader
print("dataset.py import OK")

dataset.py import OK
